### Import Depedencies

In [22]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch
from qdrant_client import QdrantClient
from qdrant_client.models import Document,Prefetch,FusionQuery

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-mpnet-base-v2", device=device)
qdrant_client = QdrantClient(host="localhost", port=6333)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### instantiate a reranker

In [5]:
reranker = CrossEncoder("BAAI/bge-reranker-base")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SURYA ER\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [6]:
def rerank(query, docs):
    pairs = [(query, d) for d in docs]
    scores = reranker.predict(pairs)

    return [
        doc for doc, _ in sorted(
            zip(docs, scores),
            key=lambda x: x[1],
            reverse=True
        )
    ]

In [8]:
def get_embeddings(text):
    
    embedded_text = embedding_model.encode(inputs=text, normalize_embeddings = True, convert_to_numpy = True)
    return embedded_text.tolist()

In [9]:
def retrieve_data(query, limit = 10):


    embedded_query = get_embeddings(query)
    results = qdrant_client.query_points(collection_name="steam-data-collection-hybrid-search", limit=limit, prefetch=[

        Prefetch(
            query=embedded_query,
            using="dense",
            limit=limit
        ),
        Prefetch(
            query=Document(text=query,model="qdrant/bm25"),
            using="sparse",
            limit=limit
        )
    ], query = FusionQuery(fusion="rrf"))

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_score = []
    retrieved_genres = []
    retrieved_names = []

    for result in results.points:

        retrieved_context_ids.append(result.payload["appid"])
        retrieved_contexts.append(result.payload["detailed_description"])
        similarity_score.append(result.score)
        retrieved_genres.append(result.payload["genres"])
        retrieved_names.append(result.payload["name"])

    return {
        "retrieved_context_ids":retrieved_context_ids,
        "retrieved_contexts":retrieved_contexts,
        "similarity_scores":similarity_score,
        "retrieved_genres":retrieved_genres,
        "retrieved_names": retrieved_names
    }



In [14]:
user_query = "suggest me multiplayer fps games"

In [16]:
result = retrieve_data(user_query)

In [17]:
result["retrieved_contexts"]

['Thrillville™: Off the Rails lives up to its name with 20 death-defying rides so outrageous, they inspire the same word from every park visitor who sees them:  “WHOA!” Players build these incredible “WHOA Coasters” to leap from one track to another, launch through the air like cannonballs, blast through a burning ring of fire and more.<br>\t\t\t\tBut the new fun doesn’t stop there. Off the Rails features 34 playable multiplayer minigames, 15 all-new theme areas, over 40 thrill rides, a new story that ties together more than 100 missions, and social interaction with park guests that’s better than ever. The in-depth conversations both advance the plot and suggest better ways to manage the park. But is every guest to be trusted?<br>\t\t\t\t<ul class="bb_ul"><li>Experience the visceral fun of interacting with a theme park you create.<br>\t\t\t\t</li><li>Build and ride your own creations, talk and joke with all your guests, and play dozens of minigames.<br>\t\t\t\t</li><li>Visit 15 themed 

In [18]:
reranked_result = rerank(user_query,result["retrieved_contexts"])

In [21]:
reranked_result

['Bringing the legendary war between two of science-fiction\'s most popular characters to FPS fans, AvP delivers three outstanding single player campaigns and provides untold hours of unique 3-way multiplayer gaming. <br><br>\t\t\t\t\tExperience distinctly new and thrilling first person gameplay as you survive, hunt and prey in the deadly jungles and swamps surrounding the damned colony of Freya\'s Prospect. <br><br>\t\t\t\t\t<ul class="bb_ul"><li>As the Marine, you\'ll experience a claustrophobic and terrifying experience where light is your friend, but there\'s never enough. However, the United States Marine Corps are humanity\'s last line of defense, and as such they are armed to the teeth with the very latest in high explosive and automatic weaponry. <br>\t\t\t\t\t\t</li><li>As the Predator, you will stalk from the shadows and from above, passing athletically through the treetops to ambush your victims. Although equipped with an array of powerful, exotic weapons and tracking equipm